# SAMU-MG · Confiabilidade dos indicadores de tempo · Entrega 1 (v2)
Lê a **prata** (colunas já separadas) ou a **bronze** (uma linha por dia, JSON em `occurrences`; `CAMADA = "bronze"`),
mantém **somente campos em lista branca** (sem nome de paciente, telefone, endereço ou nomes de agentes)
e gera tabelas agregadas para o relatório.

In [ ]:
# CONFIGURAÇÃO ---------------------------------------------------------------
CAMADA = "prata"                          # "prata" (recomendado: já vem em colunas) | "bronze" (JSON por dia)
SOURCE_PRATA_SCHEMA = "prata_dev.vskysamu"  # prata: schema com uma tabela por consórcio; o notebook une todas, só com as colunas da lista branca
SOURCE_PRATA = None                       # alternativa: uma tabela já unificada (com a coluna `tabela`); usar só se SOURCE_PRATA_SCHEMA = None
CONSORCIOS = None                         # prata: None = todos; ou lista de nomes de tabela, ex.: ["cisdeste", "consurge_gv"]
SOURCE_SCHEMA = "bronze_dev.vskysamu"     # só bronze: schema com uma tabela por consórcio (None para usar SOURCE_TYPE/SOURCE)
TABLES = None                             # None = descobre todas as tabelas do schema; ou lista, ex.: ["cisnorje", "cisdeste"]
EXCLUDE = []                              # tabelas a ignorar
SAMPLE_FRACTION = None                    # ex.: 0.10 = amostra de 10% dos DIAS (linhas da bronze), reprodutível; None = base completa
SEED = 42
DIAS_POR_CONSORCIO = None                # ex.: 30 = sorteia 30 dias de CADA consórcio (equilibra tamanhos; reprodutível); None = todos os dias
MIN_N = 200                               # abaixo disso o resultado do consórcio é marcado como amostra pequena
SOURCE_TYPE = "table"                     # usado só se SOURCE_SCHEMA = None: "table" | "parquet" | "csv"
SOURCE = "catalogo.bronze.vskysamu"
OUT_DIR = "/Workspace/Users/SEU_EMAIL/entrega1_saidas_v2"   # AJUSTAR: pasta do workspace ou Volume, para baixar os CSVs agregados pela interface
FLAT_TABLE = None                         # ex.: "catalogo.samu_quali.ocorrencias_flat" para gravar a tabela sem PII (entrada do script de anonimização)

TS_FMT = "dd/MM/yyyy HH:mm:ss"
LIMITES_MIN = {"t1": 120, "t2": 240, "t2_efetivo": 180, "t3": 480, "t3_efetivo": 240}  # PROVISÓRIOS: pactuar
MG_BOX = dict(lat_min=-23.0, lat_max=-14.0, lon_min=-52.0, lon_max=-39.0)

In [ ]:
import os
import pandas as pd
from scipy import stats
import pyspark.sql.functions as F
from pyspark.sql import Window

os.makedirs(OUT_DIR, exist_ok=True)

fontes = {}   # nome do consórcio (tabela) -> DataFrame da bronze
if CAMADA == "bronze":
    if SOURCE_SCHEMA:
        if TABLES is None:
            tabs = spark.sql(f"SHOW TABLES IN {SOURCE_SCHEMA}").collect()
            TABLES = [r.tableName for r in tabs if not r.isTemporary and r.tableName not in EXCLUDE]
        print("Tabelas (consórcios):", TABLES)
        for t in TABLES:
            fontes[t] = spark.table(f"{SOURCE_SCHEMA}.{t}")
    else:
        if SOURCE_TYPE == "table":
            fontes["unico"] = spark.table(SOURCE)
        elif SOURCE_TYPE == "parquet":
            fontes["unico"] = spark.read.parquet(SOURCE)
        else:
            fontes["unico"] = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(SOURCE)

## 1. Leitura com lista branca de campos (prata: renomeia e tipa · bronze: extrai o JSON e confere a integridade)

In [ ]:
if CAMADA == "bronze":
    declarados, partes_occ = {}, []
    for nome, df_t in fontes.items():
        if SAMPLE_FRACTION:
            df_t = df_t.sample(False, SAMPLE_FRACTION, SEED)   # amostra reprodutível de dias
        if DIAS_POR_CONSORCIO:
            df_t = df_t.orderBy(F.rand(SEED)).limit(DIAS_POR_CONSORCIO)   # cada linha da bronze = 1 dia
        if "records_in_page" in df_t.columns:
            declarados[nome] = df_t.agg(F.sum("records_in_page")).first()[0]
        occ_col = (F.from_json("occurrences", "array<map<string,string>>")
                   if dict(df_t.dtypes)["occurrences"] == "string" else F.col("occurrences"))
        partes_occ.append(df_t.select(F.col("date").alias("data_particao"), occ_col.alias("arr"))
                              .select("data_particao", F.explode("arr").alias("o"))
                              .withColumn("consorcio", F.lit(nome)))
    occ = partes_occ[0]
    for p_ in partes_occ[1:]:
        occ = occ.unionByName(p_)

    def f(name):
        c = F.trim(F.col("o")[name].cast("string"))
        return F.when(c == "", None).otherwise(c)

    def ts(name):
        return F.to_timestamp(f(name), TS_FMT)

    MARCOS = ["pj9", "pj10", "sj9", "sj10"]
    flat = occ.select(
        "data_particao", "consorcio",
        f("ID_SAMU").alias("idsamu"), f("NOME_SAMU").alias("nomesamu"),
        f("NUM_OCORRENCIA").alias("numocorrencia"), f("ID_OCORRENCIA").alias("idocorrencia"),
        f("CODIGO").alias("codigo"), f("COM_ATENDIMENTO").alias("comatendimento"), f("STATUS").alias("status"),
        f("TIPO_UNIDADE").alias("tipounidade"), f("TIPO_TRANSPORTE").alias("tipotransporte"),
        f("OBITO").alias("obito"), f("TIPO_OBITO").alias("tipoobito"),
        f("IDADE").cast("int").alias("idade"), f("SEXO").alias("sexo"),
        f("HOSPITAL_DESTINO").alias("hospitaldestino"),
        ts("DATA_CRIACAO").alias("datacriacao"), ts("DATA_TARM").alias("datatarm"),
        ts("DATA_REGULADOR").alias("dataregulador"), ts("DATA_RADIO_OPERADOR").alias("dataradiooperador"),
        ts("PJ9").alias("pj9"), ts("PJ10").alias("pj10"), ts("SJ9").alias("sj9"), ts("SJ10").alias("sj10"),
        *[f(f"LAT_{m.upper()}").cast("double").alias(f"lat_{m}") for m in MARCOS],
        *[f(f"LONG_{m.upper()}").cast("double").alias(f"long_{m}") for m in MARCOS],
        f("FREQ_CARDICA").cast("int").alias("freq_cardiaca"), f("PA_SISTOLICA").cast("int").alias("pa_sistolica"),
        f("OXIMETRIA_PULSO").cast("int").alias("oximetria"), f("GLASGOW").cast("int").alias("glasgow"),
    ).withColumn("ano", F.year("datacriacao"))

    obtidos = {r["consorcio"]: r["n"] for r in flat.groupBy("consorcio").agg(F.count("*").alias("n")).collect()}
    for nome, esperado in declarados.items():
        ob = obtidos.get(nome, 0)
        print(f"Integridade [{nome}]: declarado na paginação = {esperado} | extraído do JSON = {ob} | "
              f"{'OK' if esperado == ob else 'DIVERGE - investigar'}")

else:
    # PRATA: já vem em colunas (PascalCase). Só padroniza nomes, vazios e tipos, e mantém a lista branca.
    NEEDED = ["Date", "IdSamu", "NomeSamu", "NumOcorrencia", "IdOcorrencia", "Codigo", "ComAtendimento", "Status",
              "TipoUnidade", "TipoTransporte", "Obito", "TipoObito", "Idade", "Sexo", "HospitalDestino",
              "DataCriacao", "DataTarm", "DataRegulador", "DataRadioOperador", "Pj9", "Pj10", "Sj9", "Sj10",
              "LatPj9", "LatPj10", "LatSj9", "LatSj10", "LongPj9", "LongPj10", "LongSj9", "LongSj10",
              "FreqCardica", "PaSistolica", "OximetriaPulso", "Glasgow"]
    if SOURCE_PRATA_SCHEMA:
        tabs = [r.tableName for r in spark.sql(f"SHOW TABLES IN {SOURCE_PRATA_SCHEMA}").collect()
                if not r.isTemporary and r.tableName not in EXCLUDE]
        if CONSORCIOS:
            tabs = [t for t in tabs if t.lower() in [c.lower() for c in CONSORCIOS]]
        print("Tabelas da prata:", tabs)
        partes_p = []
        for t in tabs:
            d_t = spark.table(f"{SOURCE_PRATA_SCHEMA}.{t}")
            cols = {c.lower(): c for c in d_t.columns}
            if DIAS_POR_CONSORCIO:
                dias = d_t.select(cols["date"]).distinct().orderBy(F.rand(SEED)).limit(DIAS_POR_CONSORCIO)
                d_t = d_t.join(dias, cols["date"], "left_semi")
            d_t = d_t.select(*[F.col(cols[n.lower()]) for n in NEEDED if n.lower() in cols])  # poda antes de unir: sem PII
            partes_p.append(d_t.withColumn("tabela", F.lit(t)))
        prata = partes_p[0]
        for p_ in partes_p[1:]:
            prata = prata.unionByName(p_, allowMissingColumns=True)
    else:
        prata = spark.table(SOURCE_PRATA)
        if CONSORCIOS:
            prata = prata.where(F.lower(F.col("tabela")).isin([c.lower() for c in CONSORCIOS]))
    if SAMPLE_FRACTION:
        prata = prata.sample(False, SAMPLE_FRACTION, SEED)
    tipos = dict(prata.dtypes)

    def pc(nome):
        c = F.trim(F.col(nome).cast("string"))
        return F.when(c == "", None).otherwise(c)

    def pts(nome):
        if tipos[nome] in ("timestamp", "timestamp_ntz", "date"):
            return F.col(nome).cast("timestamp")
        return F.to_timestamp(pc(nome), TS_FMT)

    MARCOS = ["pj9", "pj10", "sj9", "sj10"]
    flat = prata.select(
        F.col("Date").cast("string").alias("data_particao"), F.lower(F.col("tabela")).alias("consorcio"),
        pc("IdSamu").alias("idsamu"), pc("NomeSamu").alias("nomesamu"),
        pc("NumOcorrencia").alias("numocorrencia"), pc("IdOcorrencia").alias("idocorrencia"),
        F.upper(pc("Codigo")).alias("codigo"), pc("ComAtendimento").alias("comatendimento"), pc("Status").alias("status"),
        pc("TipoUnidade").alias("tipounidade"), pc("TipoTransporte").alias("tipotransporte"),
        pc("Obito").alias("obito"), pc("TipoObito").alias("tipoobito"),
        pc("Idade").cast("int").alias("idade"), pc("Sexo").alias("sexo"), pc("HospitalDestino").alias("hospitaldestino"),
        pts("DataCriacao").alias("datacriacao"), pts("DataTarm").alias("datatarm"),
        pts("DataRegulador").alias("dataregulador"), pts("DataRadioOperador").alias("dataradiooperador"),
        pts("Pj9").alias("pj9"), pts("Pj10").alias("pj10"), pts("Sj9").alias("sj9"), pts("Sj10").alias("sj10"),
        *[F.col(f"Lat{m.capitalize()}").cast("double").alias(f"lat_{m}") for m in MARCOS],
        *[F.col(f"Long{m.capitalize()}").cast("double").alias(f"long_{m}") for m in MARCOS],
        F.col("FreqCardica").cast("int").alias("freq_cardiaca"), F.col("PaSistolica").cast("int").alias("pa_sistolica"),
        F.col("OximetriaPulso").cast("int").alias("oximetria"), F.col("Glasgow").cast("int").alias("glasgow"),
    ).withColumn("ano", F.year("datacriacao"))

if FLAT_TABLE:
    flat.write.mode("overwrite").saveAsTable(FLAT_TABLE)
    print("Tabela sem PII gravada em", FLAT_TABLE)

## 2. Chaves, granularidade e atendimento (1 linha por atendimento)
Chamada = `ID_SAMU + ano(DATA_CRIACAO) + NUM_OCORRENCIA` · Atendimento = `ID_SAMU + ID_OCORRENCIA`.
Nos repetidos, mantém a linha mais completa nos marcos (as linhas repetidas diferem, principalmente, nos campos de equipe).

In [ ]:
TIME_COLS = ["datacriacao", "datatarm", "dataregulador", "dataradiooperador", "pj9", "pj10", "sj9", "sj10"]
flat = (flat
        .withColumn("chave_chamada", F.concat_ws("|", "idsamu", F.col("ano").cast("string"), "numocorrencia"))
        .withColumn("chave_atend", F.concat_ws("|", "idsamu", "idocorrencia")))

granul = (flat.groupBy("consorcio").agg(
    F.count("*").alias("linhas"),
    F.countDistinct("data_particao").alias("datas_particao"),
    F.countDistinct("chave_chamada").alias("chamadas_distintas"),
    F.countDistinct("chave_atend").alias("atendimentos_distintos"),
    F.countDistinct("idsamu").alias("ids_samu"),
    F.min("nomesamu").alias("nomesamu_exemplo"),
    F.min("datacriacao").alias("data_min"),
    F.max("datacriacao").alias("data_max"),
).orderBy("consorcio").toPandas())
granul["atend_por_dia"] = (granul["atendimentos_distintos"] / granul["datas_particao"]).round(1)
granul["pct_do_total_atend"] = (100 * granul["atendimentos_distintos"] / granul["atendimentos_distintos"].sum()).round(1)
granul["amostra_pequena"] = granul["atendimentos_distintos"] < MIN_N
display(granul); granul.to_csv(f"{OUT_DIR}/01_granularidade.csv", index=False)

n_pre = sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in TIME_COLS)
w = Window.partitionBy("chave_atend").orderBy(n_pre.desc(), F.col("datacriacao").asc())
at = flat.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn").cache()
print("Atendimentos:", at.count())

## 3. Completude: bruta vs. condicionada
A baixa completude bruta vem sobretudo do denominador (trotes, quedas de ligação, orientações não têm despacho).
- Denominador A: `COM_ATENDIMENTO = 'S'` (marcos de deslocamento).
- Denominador B: `HOSPITAL_DESTINO` preenchido (marcos de transporte).

In [ ]:
nn = lambda c: F.col(c).isNotNull()
def pct(num, den):
    return F.round(100 * F.sum(F.when(den & num, 1).otherwise(0)) / F.sum(F.when(den, 1).otherwise(0)), 1)
todos, A, B = F.lit(True), (F.col("comatendimento") == "S"), nn("hospitaldestino")

comp = (at.rollup("consorcio", "ano").agg(
    F.count("*").alias("n"),
    pct(nn("dataradiooperador"), todos).alias("radio_bruto"),
    pct(nn("pj10"), todos).alias("pj10_bruto"),
    pct(nn("sj10"), todos).alias("sj10_bruto"),
    pct(nn("dataradiooperador"), A).alias("radio_dado_atend"),
    pct(nn("pj9"), A).alias("pj9_dado_atend"),
    pct(nn("pj10"), A).alias("pj10_dado_atend"),
    pct(nn("sj10"), B).alias("sj10_dado_hospital"),
).orderBy("consorcio", "ano").toPandas())
display(comp); comp.to_csv(f"{OUT_DIR}/02_completude.csv", index=False)

por_codigo = (at.groupBy("consorcio", "codigo").agg(
    F.count("*").alias("n"),
    pct(nn("dataregulador"), todos).alias("regulador"), pct(nn("dataradiooperador"), todos).alias("radio"),
    pct(nn("pj10"), todos).alias("pj10"), pct(nn("sj10"), todos).alias("sj10")).toPandas().sort_values(["consorcio", "n"], ascending=[True, False]))
display(por_codigo); por_codigo.to_csv(f"{OUT_DIR}/02b_completude_por_codigo.csv", index=False)

## 4. Regras temporais por par de marcos consecutivos (inversão, zero, < 1 min)

In [ ]:
def minutos(a, b):
    return (F.unix_timestamp(b) - F.unix_timestamp(a)) / 60.0

def rotulo(df_):
    df_["consorcio"] = df_["consorcio"].fillna("TOTAL (agregado)")
    return df_

PARES = [
    ("criacao_tarm", "datacriacao", "datatarm"),
    ("tarm_regulador", "datatarm", "dataregulador"),
    ("regulador_radio", "dataregulador", "dataradiooperador"),
    ("criacao_radio (T1)", "datacriacao", "dataradiooperador"),
    ("radio_pj9", "dataradiooperador", "pj9"),
    ("pj9_pj10 (T2 efetivo)", "pj9", "pj10"),
    ("radio_pj10 (T2)", "dataradiooperador", "pj10"),
    ("pj10_sj9", "pj10", "sj9"),
    ("sj9_sj10 (T3 efetivo)", "sj9", "sj10"),
    ("pj10_sj10 (T3)", "pj10", "sj10"),
]
blocos = []
for nome, a, b in PARES:
    x = at.withColumn("d", minutos(a, b)).where(F.col("d").isNotNull())
    r = x.rollup("consorcio").agg(
        F.count("*").alias("n"),
        F.round(100 * F.avg((F.col("d") < 0).cast("int")), 2).alias("pct_inversao"),
        F.round(100 * F.avg((F.col("d") == 0).cast("int")), 2).alias("pct_zero"),
        F.round(100 * F.avg(((F.col("d") > 0) & (F.col("d") < 1)).cast("int")), 2).alias("pct_menor_1min"),
        F.round(F.expr("percentile_approx(d, 0.5)"), 1).alias("mediana"),
        F.round(F.expr("percentile_approx(d, 0.9)"), 1).alias("p90"),
        F.round(F.expr("percentile_approx(d, 0.99)"), 1).alias("p99"),
    ).toPandas()
    r.insert(0, "par", nome)
    blocos.append(r)
pares_df = rotulo(pd.concat(blocos, ignore_index=True))
pares_df["amostra_pequena"] = pares_df["n"] < MIN_N
display(pares_df); pares_df.to_csv(f"{OUT_DIR}/03_regras_por_par.csv", index=False)

## 5. Preferência por dígito e segundos zerados (por central)
Esperado se o horário é gerado pelo sistema: minuto terminado em 0/5 ≈ 20% e segundo = 0 ≈ 1,7%.

In [ ]:
partes = [at.where(nn(c)).select("consorcio", F.lit(c).alias("campo"), (F.minute(c) % 10).alias("d"),
                                 (F.second(c) == 0).cast("int").alias("seg0")) for c in TIME_COLS]
base_dig = partes[0]
for p in partes[1:]:
    base_dig = base_dig.unionByName(p)
agg = base_dig.groupBy("consorcio", "campo").agg(
    F.count("*").alias("n"), F.sum("seg0").alias("n_seg0"),
    *[F.sum(F.when(F.col("d") == k, 1).otherwise(0)).alias(f"d{k}") for k in range(10)]).toPandas()
rows = []
for _, r in agg.iterrows():
    obs = [r[f"d{k}"] for k in range(10)]
    chi2, p = stats.chisquare(obs) if r["n"] >= 200 else (float("nan"), float("nan"))
    rows.append({"consorcio": r["consorcio"], "campo": r["campo"], "n": int(r["n"]),
                 "pct_min_0_ou_5": round(100 * (obs[0] + obs[5]) / r["n"], 1),
                 "pct_seg_zero": round(100 * r["n_seg0"] / r["n"], 1), "chi2": chi2, "p_valor": p})
digitos = pd.DataFrame(rows)
display(digitos); digitos.to_csv(f"{OUT_DIR}/04_digitos_segundos.csv", index=False)

## 6. Validação cruzada por GPS (coordenadas gravadas em PJ9, PJ10, SJ9, SJ10)
(a) coordenada nula (lat = 0) por ano · (b) deslocamento com distância ~0 mas tempo longo · (c) velocidade em linha reta implausível.

In [ ]:
def coord_valida(m):
    return (nn(f"lat_{m}") & nn(f"long_{m}") &
            F.col(f"lat_{m}").between(MG_BOX["lat_min"], MG_BOX["lat_max"]) &
            F.col(f"long_{m}").between(MG_BOX["lon_min"], MG_BOX["lon_max"]))

gps_zero = (at.groupBy("consorcio", "ano").agg(F.count("*").alias("n"),
    *[F.round(100 * F.avg(F.when(nn(m), (F.col(f"lat_{m}") == 0).cast("int"))), 1).alias(f"pct_coord_nula_{m}") for m in MARCOS]
).orderBy("consorcio", "ano").toPandas())
display(gps_zero); gps_zero.to_csv(f"{OUT_DIR}/05a_gps_coordenada_nula.csv", index=False)

def hav_km(la1, lo1, la2, lo2):
    p1, p2 = F.radians(la1), F.radians(la2)
    a = F.sin((p2 - p1) / 2) ** 2 + F.cos(p1) * F.cos(p2) * F.sin(F.radians(lo2 - lo1) / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

res = []
for nome, a, b in [("pj9_pj10", "pj9", "pj10"), ("sj9_sj10", "sj9", "sj10")]:
    ok = coord_valida(a) & coord_valida(b) & nn(a) & nn(b)
    x = (at.where(ok)
           .withColumn("km", hav_km(F.col(f"lat_{a}"), F.col(f"long_{a}"), F.col(f"lat_{b}"), F.col(f"long_{b}")))
           .withColumn("min", minutos(a, b))
           .withColumn("kmh", F.when(F.col("min") > 0, F.col("km") / (F.col("min") / 60.0))))
    r = x.rollup("consorcio").agg(F.count("*").alias("n_validos"),
              F.round(F.expr("percentile_approx(km, 0.5)"), 1).alias("mediana_km"),
              F.round(100 * F.avg((F.col("km") < 0.1).cast("int")), 1).alias("pct_dist_menor_100m"),
              F.round(100 * F.avg(((F.col("km") < 0.1) & (F.col("min") > 5)).cast("int")), 1).alias("pct_dist_100m_e_mais_5min"),
              F.round(100 * F.avg((F.col("kmh") > 110).cast("int")), 2).alias("pct_vel_maior_110"),
              F.round(100 * F.avg(((F.col("km") > 5) & (F.col("min") < 3)).cast("int")), 2).alias("pct_5km_em_menos_3min")).toPandas()
    r.insert(0, "trecho", nome)
    res.append(r)
gps_pares = pd.concat(res, ignore_index=True)
gps_pares["consorcio"] = gps_pares["consorcio"].fillna("TOTAL (agregado)")
gps_pares["amostra_pequena"] = gps_pares["n_validos"] < MIN_N
display(gps_pares); gps_pares.to_csv(f"{OUT_DIR}/05b_gps_plausibilidade.csv", index=False)

## 7. Consistência entre campos de desfecho e marcos

In [ ]:
cons = at.rollup("consorcio").agg(
    F.count("*").alias("n_atendimentos"),
    F.sum(F.when((F.col("comatendimento") == "N") & nn("dataradiooperador"), 1).otherwise(0)).alias("comatend_N_com_radio"),
    F.sum(F.when((F.col("comatendimento") == "S") & ~nn("dataradiooperador"), 1).otherwise(0)).alias("comatend_S_sem_radio"),
    F.sum(F.when(nn("hospitaldestino") & ~nn("sj10"), 1).otherwise(0)).alias("hospital_sem_sj10"),
    F.sum(F.when(nn("sj10") & ~nn("hospitaldestino"), 1).otherwise(0)).alias("sj10_sem_hospital"),
    F.sum(F.when(nn("pj10") & ~nn("dataradiooperador"), 1).otherwise(0)).alias("pj10_sem_radio"),
    F.sum(F.when((F.col("obito") == "S") & ~nn("pj10"), 1).otherwise(0)).alias("obito_S_sem_pj10"),
    F.sum(F.when(F.col("obito") == "S", 1).otherwise(0)).alias("obito_S_total"),
).toPandas()
cons = rotulo(cons)
display(cons); cons.to_csv(f"{OUT_DIR}/06_consistencia.csv", index=False)

## 8. Idade e sinais vitais: valores zero como preenchimento padrão

In [ ]:
zeros = at.rollup("consorcio").agg(
    F.count("*").alias("n"),
    F.round(100 * F.avg((F.col("idade") == 0).cast("int")), 1).alias("pct_idade_zero"),
    F.round(100 * F.avg(((F.col("idade") < 0) | (F.col("idade") > 120)).cast("int")), 1).alias("pct_idade_fora_0_120"),
    *[F.round(100 * F.avg(((F.col(c) == 0) | F.col(c).isNull()).cast("int")), 1).alias(f"pct_{c}_zero_ou_nulo")
      for c in ["freq_cardiaca", "pa_sistolica", "oximetria", "glasgow"]],
).toPandas()
zeros = rotulo(zeros)
display(zeros); zeros.to_csv(f"{OUT_DIR}/07_idade_vitais_zero.csv", index=False)

## 9. T1, T2, T3: bruto vs. válido (regras: dois marcos presentes, 0 < duração ≤ limite provisório)

In [ ]:
INDICADORES = {"t1": ("datacriacao", "dataradiooperador"), "t2": ("dataradiooperador", "pj10"),
               "t2_efetivo": ("pj9", "pj10"), "t3": ("pj10", "sj10"), "t3_efetivo": ("sj9", "sj10")}
blocos = []
for ind, (a, b) in INDICADORES.items():
    x = at.withColumn("d", minutos(a, b)).where(F.col("d").isNotNull())
    br = x.rollup("consorcio").agg(F.count("*").alias("n_bruto"), F.round(F.avg("d"), 1).alias("media_bruta"),
            F.round(F.expr("percentile_approx(d, 0.5)"), 1).alias("mediana_bruta"),
            F.round(F.expr("percentile_approx(d, 0.9)"), 1).alias("p90_bruto")).toPandas()
    va = x.where((F.col("d") > 0) & (F.col("d") <= LIMITES_MIN[ind])).rollup("consorcio").agg(
            F.count("*").alias("n_valido"), F.round(F.avg("d"), 1).alias("media_valida"),
            F.round(F.expr("percentile_approx(d, 0.5)"), 1).alias("mediana_valida"),
            F.round(F.expr("percentile_approx(d, 0.9)"), 1).alias("p90_valido")).toPandas()
    m = br.merge(va, on="consorcio", how="left")
    m.insert(0, "indicador", ind); m.insert(1, "limite_min", LIMITES_MIN[ind])
    blocos.append(m)
comp_t = rotulo(pd.concat(blocos, ignore_index=True))
comp_t["pct_descartado"] = (100 * (1 - comp_t["n_valido"] / comp_t["n_bruto"])).round(1)
comp_t["delta_media"] = (comp_t["media_valida"] - comp_t["media_bruta"]).round(1)
comp_t["amostra_pequena"] = comp_t["n_bruto"] < MIN_N
display(comp_t); comp_t.to_csv(f"{OUT_DIR}/08_t1_t2_t3_bruto_vs_valido.csv", index=False)

## 10. Resumo comparável entre consórcios (com intervalo de confiança de 95%)
Como os consórcios têm tamanhos muito diferentes, o total agregado é dominado pelos maiores. Aqui cada consórcio aparece com
o seu intervalo de Wilson (largo quando n é pequeno) e há uma linha com a **média simples entre consórcios** (cada um pesa igual).

In [ ]:
def wilson(k, n, z=1.96):
    if not n:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5) / den
    return 100 * p, 100 * (c - h), 100 * (c + h)

S = F.col("comatendimento") == "S"
H = nn("hospitaldestino")
d_rr = minutos("dataregulador", "dataradiooperador")
cnt = at.groupBy("consorcio").agg(
    F.sum(S.cast("int")).alias("n_S"),
    F.sum((S & nn("dataradiooperador")).cast("int")).alias("k_radio_S"),
    F.sum((S & nn("pj10")).cast("int")).alias("k_pj10_S"),
    F.sum(H.cast("int")).alias("n_H"),
    F.sum((H & nn("sj10")).cast("int")).alias("k_sj10_H"),
    F.sum(nn("pj10").cast("int")).alias("n_pj10"),
    F.sum((nn("pj10") & (F.col("lat_pj10") == 0)).cast("int")).alias("k_pj10_gps_nulo"),
    F.sum(d_rr.isNotNull().cast("int")).alias("n_reg_radio"),
    F.sum((d_rr < 0).cast("int")).alias("k_reg_radio_invertido"),
    F.count("*").alias("n_atend"),
    F.sum((F.col("datatarm") == F.col("datacriacao")).cast("int")).alias("k_tarm_igual_criacao"),
).toPandas()

DEF = [("radio | atendido", "k_radio_S", "n_S"), ("pj10 | atendido", "k_pj10_S", "n_S"),
       ("sj10 | hospital destino", "k_sj10_H", "n_H"), ("coord. nula no PJ10", "k_pj10_gps_nulo", "n_pj10"),
       ("regulador→rádio invertido", "k_reg_radio_invertido", "n_reg_radio"), ("TARM = criação", "k_tarm_igual_criacao", "n_atend")]
linhas = []
for _, r in cnt.iterrows():
    for nome, k, n in DEF:
        pc_, lo, hi = wilson(r[k], r[n])
        linhas.append({"consorcio": r["consorcio"], "indicador": nome, "n": int(r[n]), "pct": round(pc_, 1),
                       "ic95_inf": round(lo, 1), "ic95_sup": round(hi, 1), "amostra_pequena": int(r[n]) < MIN_N})
resumo = pd.DataFrame(linhas)
tot = cnt.drop(columns="consorcio").sum()
for nome, k, n in DEF:
    pc_, lo, hi = wilson(tot[k], tot[n])
    resumo.loc[len(resumo)] = ["TOTAL (agregado, dominado pelos maiores)", nome, int(tot[n]), round(pc_, 1), round(lo, 1), round(hi, 1), False]
macro = resumo[~resumo["consorcio"].str.startswith("TOTAL")].groupby("indicador")["pct"].mean().round(1)
for nome, v in macro.items():
    resumo.loc[len(resumo)] = ["MÉDIA SIMPLES entre consórcios", nome, None, v, None, None, False]
display(resumo); resumo.to_csv(f"{OUT_DIR}/09_resumo_comparavel_ic95.csv", index=False)

## 11. Saídas
Arquivos em `OUT_DIR`: 01 granularidade · 02 completude (+02b por código) · 03 regras por par · 04 dígitos e segundos ·
05a coordenada nula por ano · 05b plausibilidade GPS · 06 consistência · 07 idade e vitais · 08 T1/T2/T3 bruto vs válido · 09 resumo comparável com IC95.
Regras, consistência, idade/vitais e T1/T2/T3 saem por consórcio e no total agregado; linhas com `amostra_pequena = True` não devem ser interpretadas.
Todas agregadas; nenhuma contém identificador de paciente ou de profissional.